# Regularized Regression Workflow Example

This notebook demonstrates how to run the `reg_regression_toolkit` pipeline on the bundled dummy dataset. It mirrors the original BLCA analysis workflow while supporting both binary and multi-class classification.


In [ ]:
import pandas as pd

from reg_regression_toolkit import evaluation
from reg_regression_toolkit.filters import load_feature_list
from reg_regression_toolkit.reporting import (
    ensure_output_dir,
    export_shap_summaries,
    generate_curve_plots,
    save_predictions_table,
    write_config_json,
)
from reg_regression_toolkit.workflow import run_workflow


In [ ]:
expression = pd.read_csv("../data/dummy_expression.csv")
metadata = pd.read_csv("../data/dummy_metadata.csv")

remove_features = load_feature_list("../data/dummy_remove_features.txt")
keep_features = load_feature_list("../data/dummy_keep_features.txt")

expression.head()


In [ ]:
logistic_params = {
    "Cs": [0.1, 1.0],
    "l1_ratios": [0.5, 0.9],
    "cv": 3,
    "max_iter": 500,
    "n_jobs": None,
}

artifacts = run_workflow(
    expression,
    metadata_df=metadata,
    label_column="Sample_type",
    remove_features=remove_features,
    keep_features=keep_features,
    add_back_features=("feature_signal_partner",),
    cv_splits=3,
    logistic_kwargs=logistic_params,
)

artifacts.fold_metrics


In [ ]:
output_dir = ensure_output_dir("../outputs/workflow_example")
output_dir


In [ ]:
predictions = save_predictions_table(artifacts, output_dir)
predictions.head()


In [ ]:
artifacts.coefficient_summary


In [ ]:
metrics_summary = generate_curve_plots(artifacts, output_dir)
metrics_summary


In [ ]:
importance_table = export_shap_summaries(artifacts, output_dir)
importance_table


In [ ]:
config_data = {
    "inputs": {
        "expression_path": "../data/dummy_expression.csv",
        "metadata_path": "../data/dummy_metadata.csv",
        "remove_features_path": "../data/dummy_remove_features.txt",
        "keep_features_path": "../data/dummy_keep_features.txt",
    },
    "filters": {
        "remove_features": remove_features,
        "keep_features": keep_features,
        "add_back_features": ["feature_signal_partner"],
    },
    "cross_validation": {
        "splits": 3,
        "shuffle": True,
        "random_state": 42,
    },
    "model": {
        "type": "LogisticRegressionCV",
        "parameters": logistic_params,
    },
    "output_dir": str(output_dir),
}

config_path = write_config_json(config_data, output_dir)
config_path


In [ ]:
# Optional: compute a binary ROC curve by focusing on BLCA vs Control samples
binary_metadata = metadata[metadata["Sample_type"].isin(["BLCA", "Control"])].reset_index(drop=True)
binary_expression = expression[expression["sample_id"].isin(binary_metadata["sample_id"])].reset_index(drop=True)

binary_artifacts = run_workflow(
    binary_expression,
    metadata_df=binary_metadata,
    label_column="Sample_type",
    remove_features=remove_features,
    keep_features=keep_features,
    add_back_features=("feature_signal_partner",),
    cv_splits=3,
    logistic_kwargs=logistic_params,
)

roc_info = evaluation.roc_curve_from_result(binary_artifacts.result, positive_label=1)
roc_info["auc"]
